# آموزش و ارزیابی دستیار کشاورزحساب روی Kaggle

این نوت‌بوك برای **باک‌آف سه مدل** است. برای هر مدل یک بار اجرا کنید (Runtime ← GPU T4 x2).

| مدل | نقش | حجم Q4_K_M |
|---|---|---|
| `Qwen/Qwen3.5-2B` | پیش‌فرض (۲۰۱ زبان، ۲B) | ۱.۲۸GB |
| `Qwen/Qwen3-1.7B` | تور نجات (اثبات‌شده) | ۱.۱۱GB |
| `XHToken/Spark-X2.6-4B` | چالشگر (۴B، en/zh) | ۲.۶۲GB |

معیار عبور: JSON معتبر ۱۰۰٪، intent ≥ ۹۸٪، فیلدها ≥ ۹۸٪ روی `hard-test.jsonl`.
هیچ عددی را از بنچمارک سازنده نپذیرید؛ فقط `eval_report.json` همین اجرا معیار است.


In [ ]:
# @title انتخاب مدل
MODEL = "Qwen/Qwen3.5-2B"  # @param ["Qwen/Qwen3.5-2B", "Qwen/Qwen3-1.7B", "XHToken/Spark-X2.6-4B"] {allow-input: true}
EPOCHS = 3  # @param {type:"number"}
print("MODEL =", MODEL)


In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "bf16", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)


In [ ]:
!git clone -q https://github.com/havedays67-ai/kishavar-hesab.git
%cd kishavar-hesab
!pip install -q "transformers==4.57.1" "peft>=0.13" "bitsandbytes>=0.44" datasets accelerate sentencepiece


In [ ]:
import json
for split in ["train", "val", "test", "hard-test"]:
    rows = [json.loads(line) for line in open(f"training/data/{split}.jsonl", encoding="utf-8")]
    print(split, len(rows))


In [ ]:
import subprocess, sys
name = MODEL.split("/")[-1].lower().replace(".", "-")
out = f"/kaggle/working/out/{name}"
cmd = [sys.executable, "training/train_qlora.py", "--model", MODEL,
       "--data", "training/data", "--out", out,
       "--epochs", str(EPOCHS), "--batch", "1", "--grad-accum", "16"]
print(" ".join(cmd))
proc = subprocess.run(cmd, text=True)
print("exit:", proc.returncode)


In [ ]:
import json, pathlib
name = MODEL.split("/")[-1].lower().replace(".", "-")
report = json.loads(pathlib.Path(f"/kaggle/working/out/{name}/eval_report.json").read_text(encoding="utf-8"))
hard = report["hard"]
print(json.dumps({k: hard[k] for k in ("n","json_validity","intent_accuracy","field_accuracy","action_accuracy","exact_match","passed")}, ensure_ascii=False, indent=2))
print("\nper-intent:")
for intent, row in hard["per_intent"].items():
    print(f'{intent:18s} n={row["n"]:2d}  intent_recall={row["intent_recall"]:.3f}  field_acc={row["field_accuracy"]:.3f}')
print("\nfirst errors:")
for err in hard["errors"][:10]:
    print(err)


## بعد از اجرا

1. `eval_report.json` و آداپتر را از `/kaggle/working/out/...` بردارید.
2. برای هر سه مدل همین نوت‌بوك را با `MODEL` متفاوت اجرا کنید.
3. فقط برنده‌ای که آستانه‌ها را رد کند به GGUF تبدیل می‌شود:
   ```bash
   BASE_MODEL=... ADAPTER=... OUT=... bash training/export_gguf.sh
   ```
4. آداپتر/GGUF و گزارش را در HF آپلود کنید؛ کلید را در Kaggle Secret بگذارید، نه در نوت‌بوك.

### دسترسی به دیتاست از HF (اختیاری)
```python
from datasets import load_dataset
ds = load_dataset("Hamio/kishavar-hesab-dataset")
```
